# Version Mismatch & Versioned Subfolders — Integration Test

This notebook tests the versioned subfolder upgrade flow:

1. Install **old version** (`0.1.109`), create DO+DS state with a job
2. Upgrade to **current branch**
3. **DO login** — detect mismatch, upgrade (local delete only, remote preserved in version subfolders)
4. **DO re-establishes peer** + syncs on new version
5. **DS on old version** — observe what DS sees after DO upgrade, try job submission
6. **DS upgrades** — detect mismatch, upgrade, verify old version subfolders persist
7. **DS re-establishes peer** + submits job on new version
8. Verify version subfolders on GDrive + cleanup

**Prerequisites**: OAuth tokens in `credentials/` directory.

## Step 1: Install old version and create state

In [ ]:
!uv pip install syft-client==0.1.109

In [ ]:
# Restart kernel after install
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import syft_client
from syft_client.sync.syftbox_manager import SyftboxManager
print(f"Installed version: {syft_client.__version__}")

In [ ]:
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

In [ ]:
# Create DO and DS clients on old version
do_client = SyftboxManager.for_jupyter(
    email=EMAIL_DO, has_do_role=True, token_path=token_path_do
)
ds_client = SyftboxManager.for_jupyter(
    email=EMAIL_DS, has_ds_role=True, token_path=token_path_ds
)

# Set up peer relationship
ds_client.add_peer(EMAIL_DO)
do_client.load_peers()
do_client.approve_peer_request(EMAIL_DS)
ds_client.load_peers()

# Submit a job from DS to DO
ds_client.submit_bash_job(user=EMAIL_DO, script='echo "hello from old version"')
do_client.sync()
print(f"Created state with version {syft_client.__version__}")

In [ ]:
# DO processes the job
do_client.jobs[0].approve()
do_client.process_approved_jobs()
do_client.sync()
print("DO processed and synced job results")

In [ ]:
# Write version files so mismatch check has something to find
do_client.peer_manager.write_own_version()
ds_client.peer_manager.write_own_version()
print("Version files written for DO and DS (0.1.109)")

## Step 2: Upgrade to current branch

Install the dev version. `SYFT_CLIENT_VERSION` changes while GDrive still has `0.1.109` state.

In [ ]:
!uv pip install -e /Users/swag/Documents/Coding/syft-client

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import syft_client
print(f"Installed version: {syft_client.__version__}")

## Step 3: DO login with version mismatch

Choose **[1] Upgrade** — deletes local state only, old remote version subfolders preserved.

In [ ]:
from syft_client.sync.login import login_do
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

# Should detect mismatch and prompt. Choose [1] Upgrade.
do_client = login_do(
    email=EMAIL_DO,
    token_path=token_path_do,
    sync=False,
    load_peers=False,
)

In [ ]:
# Verify DO version invariant
from syft_client.gdrive_utils import read_local_version
from syft_client.version import SYFT_CLIENT_VERSION

do_remote = do_client.read_own_version()
do_local = read_local_version(do_client.syftbox_folder)

print(f"Expected:   {SYFT_CLIENT_VERSION}")
print(f"DO remote:  {do_remote.syft_client_version if do_remote else 'None'}")
print(f"DO local:   {do_local.syft_client_version if do_local else 'None'}")

## Step 4: DO re-establishes peer and syncs

DO has upgraded. Peer state persists in SYFT_peers.json. `load_peers()` eagerly creates version subfolders in DO-owned P2P folders.

In [ ]:
# Load peers — DS should still be known from SYFT_peers.json
# This also eagerly creates version subfolders in DO-owned P2P folders
do_client.load_peers()
print(f"DO approved peers: {[p.email for p in do_client.peer_manager.approved_peers]}")
print(f"DO peer requests: {[p.email for p in do_client.peer_manager.requested_by_me_peers]}")

In [ ]:
# DO syncs — this will use new version subfolders inside existing P2P folders
try:
    do_client.sync()
    print("DO sync completed on new version")
except Exception as e:
    print(f"DO sync: {type(e).__name__}: {e}")

## Step 5: DS on old version after DO upgrade

DS is still on 0.1.109. What does DS experience?
- DS still has old P2P folders (no version subfolders in 0.1.109)
- DO's new version subfolders won't be visible to old DS code
- Job submission should still technically work (writes to the old flat structure)

In [ ]:
!uv pip install syft-client==0.1.109

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import syft_client
from syft_client.sync.syftbox_manager import SyftboxManager
print(f"DS running version: {syft_client.__version__}")

from pathlib import Path
CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

In [ ]:
# Create DS client on old version
ds_client_old = SyftboxManager.for_jupyter(
    email=EMAIL_DS, has_ds_role=True, token_path=token_path_ds
)

In [ ]:
ds_client_old.load_peers()
print(f"DS peers: {ds_client_old.peer_manager.approved_peers}")

# Can DS read DO's version?
peer_version = ds_client_old.peer_manager.load_peer_version(EMAIL_DO)
print(f"DO version as seen by DS: {peer_version}")

In [ ]:
# Can DS submit a job on old version? (writes to flat P2P folder, not version subfolder)
try:
    ds_client_old.submit_bash_job(user=EMAIL_DO, script='echo "test from old DS after DO upgrade"')
    print("Job submitted (but DO won't see it - different folder structure)")
except Exception as e:
    print(f"Job submission failed: {type(e).__name__}: {e}")

In [ ]:
# DS tries to sync
try:
    ds_client_old.sync()
    print("DS sync completed")
except Exception as e:
    print(f"DS sync: {type(e).__name__}: {e}")

## Step 6: DS upgrades

Choose **[1] Upgrade** — deletes local state, old remote P2P folders preserved (automatic archiving via version subfolders).

In [ ]:
!uv pip install -e /Users/swag/Documents/Coding/syft-client

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import syft_client
print(f"Installed version: {syft_client.__version__}")

In [ ]:
from syft_client.sync.login import login_ds
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

# Should detect mismatch. Choose [1] Upgrade.
ds_client = login_ds(
    email=EMAIL_DS,
    token_path=token_path_ds,
    sync=False,
    load_peers=False,
)

In [ ]:
# Verify DS version invariant
from syft_client.gdrive_utils import read_local_version
from syft_client.version import SYFT_CLIENT_VERSION

ds_remote = ds_client.read_own_version()
ds_local = read_local_version(ds_client.syftbox_folder)

print(f"Expected:   {SYFT_CLIENT_VERSION}")
print(f"DS remote:  {ds_remote.syft_client_version if ds_remote else 'None'}")
print(f"DS local:   {ds_local.syft_client_version if ds_local else 'None'}")

## Step 7a: DS re-establishes peer and submits job on new version

In [ ]:
# DS submits a job on new version (writes to version subfolder inside P2P inbox)
try:
    ds_client.submit_bash_job(user=EMAIL_DO, script='echo "hello from upgraded DS"')
    print("Job submitted on new version (using version subfolder)")
except Exception as e:
    print(f"Job submission failed: {type(e).__name__}: {e}")

## Step 7b: DO recieves and approves the new job

In [ ]:
from syft_client.sync.login import login_do
from pathlib import Path

CREDENTIALS_DIR = Path("/Users/swag/Documents/Coding/syft-client/credentials")
EMAIL_DO = "sameer@openmined.org"
EMAIL_DS = "som.wom.beach@gmail.com"
token_path_do = CREDENTIALS_DIR / "token_do.json"
token_path_ds = CREDENTIALS_DIR / "token_ds.json"

# Should detect mismatch and prompt. Choose [1] Upgrade.
do_client = login_do(
    email=EMAIL_DO,
    token_path=token_path_do,
    sync=False,
    load_peers=False,
)

In [ ]:
do_client.jobs

In [ ]:
# DO processes the job
do_client.jobs[0].approve()
do_client.process_approved_jobs()
do_client.sync()
print("DO processed and synced job results")

## Step 7c: DS verifies the job output

In [ ]:
ds_client.jobs[0].stdout

## Step 8: Verify version subfolders on GDrive + Cleanup

Check that version subfolders exist inside P2P folders on both DO's and DS's drives.

In [ ]:
from google.oauth2.credentials import Credentials
from syft_client.sync.connections.drive.gdrive_transport import (
    GDriveConnection, SCOPES, build_drive_service, GDRIVE_P2P_FOLDER_DATASITE_PREFIX,
    GOOGLE_FOLDER_MIME_TYPE,
)
from syft_client.sync.connections.drive.gdrive_retry import execute_with_retries

# Check DS's drive for P2P folders with version subfolders
creds_ds = Credentials.from_authorized_user_file(str(token_path_ds), SCOPES)
service_ds = build_drive_service(creds_ds)

# Find all P2P folders on DS's drive
query = (
    f"name contains '{GDRIVE_P2P_FOLDER_DATASITE_PREFIX}'"
    f" and mimeType='{GOOGLE_FOLDER_MIME_TYPE}'"
    " and 'me' in owners and trashed=false"
)
results = execute_with_retries(service_ds.files().list(q=query, fields="files(id, name)"))
p2p_folders = results.get("files", [])

print("P2P folders on DS's drive:")
for folder in p2p_folders:
    sub_query = f"'{folder['id']}' in parents and mimeType='{GOOGLE_FOLDER_MIME_TYPE}' and trashed=false"
    sub_results = execute_with_retries(service_ds.files().list(q=sub_query, fields="files(name)"))
    subfolders = [f["name"] for f in sub_results.get("files", [])]
    print(f"  {folder['name']}")
    print(f"    Version subfolders: {subfolders}")

In [ ]:
# Also check DO's drive
creds_do = Credentials.from_authorized_user_file(str(token_path_do), SCOPES)
service_do = build_drive_service(creds_do)

results = execute_with_retries(service_do.files().list(q=query, fields="files(id, name)"))
p2p_folders_do = results.get("files", [])

print("P2P folders on DO's drive:")
for folder in p2p_folders_do:
    sub_query = f"'{folder['id']}' in parents and mimeType='{GOOGLE_FOLDER_MIME_TYPE}' and trashed=false"
    sub_results = execute_with_retries(service_do.files().list(q=sub_query, fields="files(name)"))
    subfolders = [f["name"] for f in sub_results.get("files", [])]
    print(f"  {folder['name']}")
    print(f"    Version subfolders: {subfolders}")

## Cleanup

Delete all test state from both drives.

In [ ]:
from syft_client.gdrive_utils import delete_syftbox

delete_syftbox(token_path=token_path_do, email=EMAIL_DO, verbose=True)
delete_syftbox(token_path=token_path_ds, email=EMAIL_DS, verbose=True)
print("Cleanup complete!")